# Case 03 — Chon feature bang **NHIEU phuong phap** (Filter / Wrapper / Embedded)  ·  *Bai L5*

## Y tuong
Bai L5 day 3 nhom phuong phap chon feature. Thay vi chon bang cam tinh, ta chay **nhieu phuong phap**
roi lay **dong thuan (consensus)** — feature nao duoc nhieu phuong phap giu thi cang dang tin:

| Nhom | Ban chat | Phuong phap thu o day |
|---|---|---|
| **Filter** | Cham diem tung feature, doc lap voi mo hinh (nhanh, tho) | VarianceThreshold · High-correlation · Mutual Information · ANOVA F-test |
| **Wrapper** | Thu nhieu to hop feature, huan luyen mo hinh cho tung to hop (cham, chinh xac) | RFE · RFECV · Sequential Forward |
| **Embedded** | Mo hinh TU chon feature trong luc hoc | Lasso L1 · RandomForest MDI · Permutation |

## Vi sao chay thu nghiem nay
Phan 6 notebook chinh loai feature bang cam nhan + vai chi so. O day ta chay **10 phuong phap** roi cho
chung **bo phieu**, xac nhan lai bo feature mot cach co he thong (dung ngon ngu Filter/Wrapper/Embedded).

In [1]:
# ============================================================================
# PRELUDE DUNG CHUNG cho moi notebook trong thu muc nay.
# Muc dich: moi file TU CHAY DUOC ma khong phu thuoc notebook chinh.
#   - Nap train.csv (Day chuyen A) + test.csv (Day chuyen B)
#   - Tao lai dung 4 feature co che nhu notebook chinh (de ket qua nhat quan)
# ============================================================================
import os, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 110
RANDOM_STATE = 42; np.random.seed(RANDOM_STATE)   # co dinh seed -> chay lai ra so giong nhau

# --- Nap du lieu: thu nhieu duong dan vi notebook nam trong thu muc con ---
CANDIDATES = ['../Data_Final/Data_Final','../Data_Final','Data_Final/Data_Final',
              '../../Data_Final/Data_Final','Data_Final','.']
DATA_DIR = next((c for c in CANDIDATES
                 if os.path.exists(os.path.join(c,'train.csv'))
                 and os.path.exists(os.path.join(c,'test.csv'))), None)
assert DATA_DIR is not None, 'Khong tim thay train.csv/test.csv'
train = pd.read_csv(os.path.join(DATA_DIR,'train.csv'))
test  = pd.read_csv(os.path.join(DATA_DIR,'test.csv'))

NUM = ['nhiet_do_moi_truong','nhiet_do_quy_trinh','toc_do_quay','momen_xoan','do_mon_dao']  # 5 bien so goc
CAT = ['loai_san_pham','ca_lam_viec']   # 2 bien phan loai goc
TARGET = 'hong_hoc'                      # nhan: 1 = hong trong ca ke tiep

# --- Feature Engineering theo CO CHE vat ly (giong het notebook chinh) ---
def add_features(df):
    d = df.copy()
    # (1) Chenh lech nhiet: co che tan nhiet kem HDF. Lay HIEU nen triet tieu offset khi hau -> giam shift.
    d['chenh_lech_nhiet'] = d['nhiet_do_quy_trinh'] - d['nhiet_do_moi_truong']
    # (2) Cong suat co (W) = momen(Nm) * van_toc_goc(rad/s); van_toc_goc = rpm*2pi/60. Co che qua tai cong suat PWF.
    d['cong_suat_co']     = d['momen_xoan'] * d['toc_do_quay'] * 2*np.pi/60.0
    # (3) Tich mon*momen: co che qua tai cang thang OSF (mon cang nhieu + momen cang lon -> cang de gay).
    d['tich_mon_momen']   = d['do_mon_dao'] * d['momen_xoan']
    # (4) osf_margin: khoang cach toi NGUONG OSF, nguong phu thuoc HANG SP (L/M/H). >0 = da vuot nguong.
    g = d['loai_san_pham'].map({'L':11000,'M':12000,'H':13000})
    d['osf_margin']       = d['tich_mon_momen'] - g
    return d

train_fe = add_features(train); test_fe = add_features(test)
y_train = train_fe[TARGET].values
y_test  = test_fe[TARGET].values

# Bo feature CUOI da chot o notebook chinh: 9 bien so + 1 bien phan loai (loai_san_pham)
FINAL_NUM = NUM + ['chenh_lech_nhiet','cong_suat_co','tich_mon_momen','osf_margin']
FINAL_CAT = ['loai_san_pham']
print('Train:', train.shape, '| Test:', test.shape,
      '| ti le hong train/test:', round(y_train.mean(),3), round(y_test.mean(),3))
print('FINAL_NUM =', FINAL_NUM)

Train: (14000, 8) | Test: (6000, 8) | ti le hong train/test: 0.074 0.08
FINAL_NUM = ['nhiet_do_moi_truong', 'nhiet_do_quy_trinh', 'toc_do_quay', 'momen_xoan', 'do_mon_dao', 'chenh_lech_nhiet', 'cong_suat_co', 'tich_mon_momen', 'osf_margin']


### 0) Chuan bi ma tran feature (9 so + one-hot loai_san_pham -> 11 cot)

In [2]:
from sklearn.preprocessing import StandardScaler
# One-hot loai_san_pham -> loai_M, loai_H (bo L lam goc) de moi phuong phap deu co TEN feature ro rang
d_tr = train_fe.copy(); d_te = test_fe.copy()
for lv in ['M','H']:
    d_tr[f'loai_{lv}'] = (d_tr['loai_san_pham']==lv).astype(int)
    d_te[f'loai_{lv}'] = (d_te['loai_san_pham']==lv).astype(int)
FEATS = FINAL_NUM + ['loai_M','loai_H']                 # 11 feature ung vien
Xtr_raw = d_tr[FEATS].values.astype(float); Xte_raw = d_te[FEATS].values.astype(float)
sc  = StandardScaler().fit(Xtr_raw)                     # fit CHI tren train (chong ro ri)
Xtr = sc.transform(Xtr_raw); Xte = sc.transform(Xte_raw)
sel = pd.DataFrame(index=FEATS)                         # bang tong hop: moi cot = 1 phuong phap, 1=giu / 0=loai
print('So feature ung vien:', len(FEATS)); print(FEATS)

So feature ung vien: 11
['nhiet_do_moi_truong', 'nhiet_do_quy_trinh', 'toc_do_quay', 'momen_xoan', 'do_mon_dao', 'chenh_lech_nhiet', 'cong_suat_co', 'tich_mon_momen', 'osf_margin', 'loai_M', 'loai_H']


### 1) FILTER — cham diem tung feature (doc lap mo hinh)

In [3]:
from sklearn.feature_selection import (VarianceThreshold, mutual_info_classif, f_classif)

# 1a. Low variance: loai feature gan nhu HANG SO (khong mang thong tin). Xet tren du lieu THO (truoc scale).
vt = VarianceThreshold(threshold=1e-8).fit(Xtr_raw)
sel['filter_variance'] = vt.get_support().astype(int)

# 1b. High correlation: trong cac cap |corr|>0.9 (trung thong tin), bo feature co |corr voi target| THAP hon
corr = np.corrcoef(Xtr_raw, rowvar=False)
tgt  = np.array([abs(np.corrcoef(Xtr_raw[:,i], y_train)[0,1]) for i in range(len(FEATS))])
drop = set()
for i in range(len(FEATS)):
    for j in range(i+1, len(FEATS)):
        if abs(corr[i,j]) > 0.9:
            drop.add(i if tgt[i] < tgt[j] else j)      # giu feature lien quan target hon
sel['filter_highcorr'] = [0 if i in drop else 1 for i in range(len(FEATS))]
print('Cap |corr|>0.9 -> loai:', [FEATS[i] for i in drop])

# 1c. Mutual Information: bat CA quan he PHI TUYEN (khac Pearson chi bat tuyen tinh). Giu top-8.
mi = mutual_info_classif(Xtr, y_train, random_state=RANDOM_STATE)
sel['filter_MI'] = (pd.Series(mi, index=FEATS).rank(ascending=False) <= 8).astype(int).values

# 1d. ANOVA F-test: kiem dinh trung binh feature khac nhau giua 2 lop hong/khong. Giu top-8.
F,_ = f_classif(Xtr, y_train)
sel['filter_Ftest'] = (pd.Series(F, index=FEATS).rank(ascending=False) <= 8).astype(int).values

# Bang diem 3 goc nhin (MI phi tuyen · F tuyen tinh · |corr| voi target)
display(pd.DataFrame({'MI':np.round(mi,3),'F':np.round(F,1),'|corr_target|':np.round(tgt,3)}, index=FEATS)
        .sort_values('MI', ascending=False))

Cap |corr|>0.9 -> loai: ['osf_margin', 'momen_xoan']


,MI,F,|corr_target|
do_mon_dao,0.063,553.3,0.195
osf_margin,0.030,470.2,0.180
tich_mon_momen,0.026,491.6,0.184
cong_suat_co,0.024,8.6,0.025
momen_xoan,0.019,0.6,0.006
chenh_lech_nhiet,0.009,82.0,0.076
toc_do_quay,0.006,61.0,0.066
loai_M,0.002,2.5,0.013
nhiet_do_quy_trinh,0.001,15.3,0.033
nhiet_do_moi_truong,0.000,0.0,0.001


### 2) WRAPPER — thu to hop feature, do bang chinh mo hinh

In [4]:
from sklearn.feature_selection import RFE, RFECV, SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

lr = LogisticRegression(max_iter=2000, class_weight='balanced')
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight='balanced',
                            random_state=RANDOM_STATE, n_jobs=-1)

# 2a. RFE: loai DAN feature yeu nhat (theo he so LogReg) cho den khi con 8
sel['wrap_RFE'] = RFE(lr, n_features_to_select=8).fit(Xtr, y_train).get_support().astype(int)
# 2b. RFECV: nhu RFE nhung TU chon so feature toi uu bang cross-validation (scoring AUC-PR)
rfecv = RFECV(rf, step=1, cv=skf, scoring='average_precision',
              min_features_to_select=4, n_jobs=-1).fit(Xtr, y_train)
sel['wrap_RFECV'] = rfecv.get_support().astype(int)
print('RFECV tu chon', rfecv.n_features_, 'feature (toi uu AUC-PR qua CV)')
# 2c. Sequential Forward: THEM dan feature tot nhat tung buoc cho den khi du 8
sfs = SequentialFeatureSelector(lr, n_features_to_select=8, direction='forward',
                                scoring='average_precision', cv=skf, n_jobs=-1).fit(Xtr, y_train)
sel['wrap_SFS_fwd'] = sfs.get_support().astype(int)

RFECV tu chon 10 feature (toi uu AUC-PR qua CV)


### 3) EMBEDDED — mo hinh tu chon feature khi hoc

In [5]:
from sklearn.inspection import permutation_importance
# 3a. Lasso (L1): phat tri tuyet doi -> EP he so feature yeu ve 0 => feature do bi loai
l1 = LogisticRegression(penalty='l1', solver='saga', C=0.1, max_iter=5000,
                        class_weight='balanced').fit(Xtr, y_train)
sel['embed_LassoL1'] = (np.abs(l1.coef_[0]) > 1e-6).astype(int)
print('L1 giu lai', int(sel['embed_LassoL1'].sum()), '/', len(FEATS), 'feature (con lai he so=0)')

# 3b. RandomForest MDI: do quan trong = tong muc giam Gini khi chia theo feature do. Giu top-8.
rf.fit(Xtr, y_train)
sel['embed_RF_MDI'] = (pd.Series(rf.feature_importances_, index=FEATS).rank(ascending=False)<=8).astype(int).values
# 3c. Permutation: xao tron 1 feature roi xem AUC-PR TUT bao nhieu (do tren TEST B). Giu feature co dong gop >0.
perm = permutation_importance(rf, Xte, y_test, scoring='average_precision',
                              n_repeats=8, random_state=RANDOM_STATE, n_jobs=-1)
sel['embed_Perm'] = (perm.importances_mean > 1e-4).astype(int)

L1 giu lai 9 / 11 feature (con lai he so=0)


### 4) TONG HOP — bang bo phieu dong thuan (consensus)

In [6]:
sel['SO_PHIEU'] = sel.sum(axis=1)               # moi feature: bao nhieu phuong phap giu lai
sel_sorted = sel.sort_values('SO_PHIEU', ascending=False)
display(sel_sorted)
n_methods = sel.shape[1] - 1                    # tru cot SO_PHIEU
print(f'\nTong so phuong phap = {n_methods}')
keep = sel_sorted.index[sel_sorted['SO_PHIEU'] >= n_methods*0.6].tolist()   # nguong dong thuan 60%
print('Feature dong thuan cao (>=60% phuong phap giu):'); print(' ', keep)

,filter_variance,filter_highcorr,filter_MI,filter_Ftest,wrap_RFE,wrap_RFECV,wrap_SFS_fwd,embed_LassoL1,embed_RF_MDI,embed_Perm,SO_PHIEU
tich_mon_momen,1,1,1,1,1,1,1,1,1,1,10
cong_suat_co,1,1,1,1,1,1,1,1,1,1,10
do_mon_dao,1,1,1,1,1,1,1,1,1,1,10
osf_margin,1,0,1,1,1,1,1,1,1,1,9
chenh_lech_nhiet,1,1,1,1,1,1,0,1,1,1,9
toc_do_quay,1,1,1,1,1,1,0,1,1,1,9
momen_xoan,1,0,1,0,1,1,1,1,1,1,8
loai_M,1,1,1,1,0,1,1,1,0,1,8
nhiet_do_moi_truong,1,1,0,0,0,1,1,1,0,1,6
nhiet_do_quy_trinh,1,1,0,1,0,1,0,0,1,1,6



Tong so phuong phap = 10
Feature dong thuan cao (>=60% phuong phap giu):
  ['tich_mon_momen', 'cong_suat_co', 'do_mon_dao', 'osf_margin', 'chenh_lech_nhiet', 'toc_do_quay', 'momen_xoan', 'loai_M', 'nhiet_do_moi_truong', 'nhiet_do_quy_trinh']


### 5) KIEM CHUNG — F1/AUC-PR: bo dong-thuan vs bo day du

In [7]:
from sklearn.metrics import f1_score, average_precision_score
from sklearn.model_selection import cross_val_predict

def eval_cols(cols, label):
    rf2 = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced',
                                 random_state=RANDOM_STATE, n_jobs=-1)
    idx = [FEATS.index(c) for c in cols]
    # nguong toi uu F1 lay tu CV TREN TRAIN (out-of-fold), khong nhin test -> chong ro ri
    oof = cross_val_predict(rf2, Xtr[:,idx], y_train, cv=skf, method='predict_proba', n_jobs=-1)[:,1]
    ts = np.linspace(0.05,0.9,80); thr = ts[int(np.argmax([f1_score(y_train,(oof>=t)) for t in ts]))]
    rf2.fit(Xtr[:,idx], y_train); p = rf2.predict_proba(Xte[:,idx])[:,1]
    return {'bo':label,'n':len(cols),'AUC_PR':average_precision_score(y_test,p),'F1':f1_score(y_test,(p>=thr))}

rows = [eval_cols(FEATS,'Day du (11)'),
        eval_cols(keep,'Dong thuan (>=60%)'),
        eval_cols([c for c in FEATS if c in sel_sorted.index[:6]],'Top-6 phieu')]
display(pd.DataFrame(rows).set_index('bo').round(4))

,n,AUC_PR,F1
bo,,,
Day du (11),11,0.6968,0.7818
Dong thuan (>=60%),10,0.6912,0.7818
Top-6 phieu,6,0.6853,0.7818


> ### 🔎 Doc ket qua (Case 03)
> - **Cac feature co che duoc da so phuong phap cung giu**: `do_mon_dao`, `osf_margin`, `tich_mon_momen`,
>   `cong_suat_co` — dong thuan cao, dung nhu ky vong domain.
> - Bo loc high-correlation tu danh dau cap **`osf_margin ↔ momen_xoan`** trung thong tin — chinh la
>   "diem cai cam" tuong tac trong du lieu.
> - **Ket qua manh nhat** (bang kiem chung):
>
> | Bo feature | So feature | F1 |
> |---|---|---|
> | Day du | 11 | **0.782** |
> | Dong thuan (≥60%) | 10 | **0.782** |
> | Top-6 phieu | 6 | **0.782** |
>
> Lay **Top-6 feature cho F1 y het bo 11** -> hon nua so feature la **du thua**. Co the dung bo gon 6 feature
> ma khong mat diem, mo hinh don gian & de giai thich hon.
>
> ### ✅ Ket luan Case 03
> Day la cach chon feature **CO PHUONG PHAP** (Filter/Wrapper/Embedded) dung ngon ngu bai L5, thay vi cam tinh.
> No vua xac nhan bo feature cuoi, vua chi ra co the tinh gon them ma khong hai F1.